In [122]:
import os
import os.path as op
from collections import OrderedDict
import pandas as pd
import numpy as np
import shutil
import matplotlib.pyplot as plt
import seaborn as sns

In [123]:
# Define the main directory and target directory paths
phyhealth_dir = "./dset/physical-health"
deriv_dir = "./derivatives/none-reduced/regression"


In [124]:
phyhealth_df = pd.read_csv(os.path.join(deriv_dir, "phyhealth-reg.csv"))

In [125]:
phyhealth_df

,src_subject_id,mctq_sdweek_calc,mctq_msfsc_calc,resp_wheeze_yn_y,resp_pmcough_yn_y,resp_diagnosis_yn_y,resp_bronch_yn_y,blood_pressure_sys_mean,blood_pressure_dia_mean,physical_activity1_y,cbcl_scr_syn_internal_t,cbcl_scr_syn_external_t,BMI
0,NDAR_INV030W95VP,9.3677,28.7685,0.0,0.0,0.0,0.0,105.500000,66.000000,2.0,62.0,40.0,429.687500
1,NDAR_INV0DC9BJZK,6.7774,26.8970,0.0,0.0,0.0,0.0,103.000000,51.000000,7.0,63.0,34.0,317.068754
2,NDAR_INV0DKWEM1A,7.8534,2.8764,0.0,0.0,0.0,0.0,132.000000,71.333333,2.0,34.0,40.0,436.890618
3,NDAR_INV0MPBK7TU,8.8526,6.7528,0.0,0.0,0.0,0.0,95.666667,62.666667,3.0,52.0,51.0,256.238495
4,NDAR_INV0RHLKA9M,9.8785,28.9406,0.0,0.0,0.0,0.0,120.500000,77.500000,7.0,66.0,64.0,329.887152
...,...,...,...,...,...,...,...,...,...,...,...,...,...
166,NDAR_INVZM2Y9JCA,8.1008,29.0518,0.0,0.0,0.0,0.0,110.333333,55.000000,2.0,40.0,48.0,195.637119
167,NDAR_INVZP49GXF4,7.6798,31.8566,0.0,0.0,0.0,0.0,108.666667,56.000000,7.0,47.0,48.0,298.438935
168,NDAR_INVZR9NMJBR,9.0393,28.5280,0.0,0.0,0.0,0.0,102.333333,63.333333,5.0,44.0,34.0,239.445495
169,NDAR_INVZT1J0KUC,7.7963,27.9828,0.0,0.0,0.0,0.0,128.500000,75.500000,3.0,50.0,44.0,322.265625


In [126]:

print("Unique subject IDs:", phyhealth_df["src_subject_id"].nunique())

Unique subject IDs: 171


In [127]:
abcd_anthro_df = pd.read_csv(os.path.join(phyhealth_dir, "ph_y_anthro.csv"))


In [128]:
abcd_anthro_df

,src_subject_id,eventname,anthro_1_height_in,anthro2heightin,anthro3heightin,anthroheightcalc,anthroweightcast,anthroweight1lb,anthroweight2lb,anthroweight3lb,anthroweightcalc,anthro_waist_cm,anthro_timestamp
0,NDAR_INV003RTV85,baseline_year_1_arm_1,56.5,56.5,NaN,56.5,0.0,93.0,93.0,NaN,93.00,31.0,2018-10-01 14:16
1,NDAR_INV003RTV85,1_year_follow_up_y_arm_1,58.5,58.5,NaN,58.5,0.0,106.0,106.0,NaN,106.00,28.0,2019-09-16 10:13
2,NDAR_INV003RTV85,3_year_follow_up_y_arm_1,60.0,NaN,NaN,60.0,NaN,120.0,NaN,NaN,NaN,NaN,2021-08-04 08:45
3,NDAR_INV005V6D2C,baseline_year_1_arm_1,56.5,56.5,NaN,56.5,0.0,100.0,100.0,NaN,100.00,30.5,2018-04-22 13:08
4,NDAR_INV005V6D2C,1_year_follow_up_y_arm_1,58.9,58.9,NaN,58.9,0.0,120.1,120.1,NaN,120.10,31.5,2019-02-09 13:20
...,...,...,...,...,...,...,...,...,...,...,...,...,...
40519,NDAR_INVZZZNB0XC,4_year_follow_up_y_arm_1,55.0,55.0,NaN,55.0,0.0,92.5,92.6,NaN,92.55,30.5,2021-02-05 17:39
40520,NDAR_INVZZZP87KR,baseline_year_1_arm_1,59.5,59.5,NaN,59.5,0.0,123.0,123.0,NaN,123.00,32.0,2017-08-04 09:46
40521,NDAR_INVZZZP87KR,1_year_follow_up_y_arm_1,62.0,62.0,NaN,62.0,0.0,152.0,152.0,NaN,152.00,35.0,2018-08-10 12:23
40522,NDAR_INVZZZP87KR,2_year_follow_up_y_arm_1,63.5,63.5,NaN,63.5,NaN,164.0,164.0,NaN,164.00,38.5,2019-08-02 11:14


In [129]:
abcd_anthro_df = abcd_anthro_df[abcd_anthro_df["src_subject_id"].isin(phyhealth_df["src_subject_id"])]
abcd_anthro_df = abcd_anthro_df[abcd_anthro_df["eventname"].isin(["2_year_follow_up_y_arm_1", "4_year_follow_up_y_arm_1"])]
abcd_anthro_df = abcd_anthro_df.dropna(subset=["anthroweightcalc"])

In [130]:
abcd_anthro_df = abcd_anthro_df[
    abcd_anthro_df.groupby("src_subject_id")["eventname"].transform(lambda x: {"2_year_follow_up_y_arm_1", "4_year_follow_up_y_arm_1"}.issubset(set(x)))
]

In [131]:
print(abcd_anthro_df)

         src_subject_id                 eventname  anthro_1_height_in  \
135    NDAR_INV030W95VP  2_year_follow_up_y_arm_1                63.0   
137    NDAR_INV030W95VP  4_year_follow_up_y_arm_1                64.0   
594    NDAR_INV0DC9BJZK  2_year_follow_up_y_arm_1                64.5   
595    NDAR_INV0DC9BJZK  4_year_follow_up_y_arm_1                67.5   
607    NDAR_INV0DKWEM1A  2_year_follow_up_y_arm_1                61.5   
...                 ...                       ...                 ...   
40219  NDAR_INVZR9NMJBR  4_year_follow_up_y_arm_1                69.0   
40246  NDAR_INVZT1J0KUC  2_year_follow_up_y_arm_1                61.2   
40247  NDAR_INVZT1J0KUC  4_year_follow_up_y_arm_1                64.0   
40458  NDAR_INVZYRTFYRP  2_year_follow_up_y_arm_1                56.0   
40459  NDAR_INVZYRTFYRP  4_year_follow_up_y_arm_1                62.0   

       anthro2heightin  anthro3heightin  anthroheightcalc  anthroweightcast  \
135              63.00              NaN     

In [132]:
print("Unique subject IDs:", abcd_anthro_df["src_subject_id"].nunique())

Unique subject IDs: 171


In [133]:
abcd_anthro_df = abcd_anthro_df[["src_subject_id", "eventname", "anthroweightcalc"]]

In [134]:
# Compute weight change from Year 2 to Year 4 and categorize participants

y2_event = "2_year_follow_up_y_arm_1"
y4_event = "4_year_follow_up_y_arm_1"
weight_col = "anthroweightcalc"

# Wide format: one row per participant with Y2 and Y4 weights
wide = abcd_anthro_df.pivot_table(
    index="src_subject_id",
    columns="eventname",
    values=weight_col,
    aggfunc="mean",
)

# Delta weight (Y4 - Y2)
delta = wide[y4_event] - wide[y2_event]

mu = delta.mean()
sigma = delta.std()

# Categorize using +/- 1 SD from the mean
conditions = [delta > (mu + sigma), delta < (mu - sigma)]
choices = ["gain", "loss"]

delta_cat = pd.Series(np.select(conditions, choices, default="stable"), index=delta.index)

# Merge category back onto the long-form dataframe
abcd_anthro_df = abcd_anthro_df.merge(
    delta_cat.rename("delta_weight").reset_index(),
    on="src_subject_id",
    how="left",
)

print("Delta weight category counts:")
print(abcd_anthro_df[["src_subject_id", "delta_weight"]].drop_duplicates()["delta_weight"].value_counts())


Delta weight category counts:
delta_weight
stable    122
gain       25
loss       24
Name: count, dtype: int64


In [135]:
# Merge delta_weight category onto phyhealth_df (one row per participant)
delta_weight_df = (
    abcd_anthro_df[["src_subject_id", "delta_weight"]]
    .drop_duplicates(subset=["src_subject_id"])
    .copy()
)

phyhealth_df = phyhealth_df.merge(delta_weight_df, on="src_subject_id", how="left")

# Ensure delta_weight is the last column
if "delta_weight" in phyhealth_df.columns:
    cols = [c for c in phyhealth_df.columns if c != "delta_weight"] + ["delta_weight"]
    phyhealth_df = phyhealth_df[cols]

print("phyhealth_df columns (last should be delta_weight):")
print(phyhealth_df.columns.tolist())
print("\nDelta weight category counts in phyhealth_df:")
print(phyhealth_df["delta_weight"].value_counts(dropna=False))

phyhealth_df columns (last should be delta_weight):
['src_subject_id', 'mctq_sdweek_calc', 'mctq_msfsc_calc', 'resp_wheeze_yn_y', 'resp_pmcough_yn_y', 'resp_diagnosis_yn_y', 'resp_bronch_yn_y', 'blood_pressure_sys_mean', 'blood_pressure_dia_mean', 'physical_activity1_y', 'cbcl_scr_syn_internal_t', 'cbcl_scr_syn_external_t', 'BMI', 'delta_weight']

Delta weight category counts in phyhealth_df:
delta_weight
stable    122
gain       25
loss       24
Name: count, dtype: int64


In [136]:
# Replace systolic/diastolic BP with weighted mean BP

sys_col = "blood_pressure_sys_mean"
dia_col = "blood_pressure_dia_mean"
mean_col = "blood_pressure_mean"

if sys_col in phyhealth_df.columns and dia_col in phyhealth_df.columns:
    sys = pd.to_numeric(phyhealth_df[sys_col], errors="coerce")
    dia = pd.to_numeric(phyhealth_df[dia_col], errors="coerce")

    phyhealth_df[mean_col] = (1/3) * sys + (2/3) * dia
    phyhealth_df = phyhealth_df.drop(columns=[sys_col, dia_col])

    print(f"Created {mean_col} and dropped {sys_col}, {dia_col}")
else:
    print(f"Warning: missing {sys_col} and/or {dia_col} in phyhealth_df")

print(phyhealth_df.head())

Created blood_pressure_mean and dropped blood_pressure_sys_mean, blood_pressure_dia_mean
     src_subject_id  mctq_sdweek_calc  mctq_msfsc_calc  resp_wheeze_yn_y  \
0  NDAR_INV030W95VP            9.3677          28.7685               0.0   
1  NDAR_INV0DC9BJZK            6.7774          26.8970               0.0   
2  NDAR_INV0DKWEM1A            7.8534           2.8764               0.0   
3  NDAR_INV0MPBK7TU            8.8526           6.7528               0.0   
4  NDAR_INV0RHLKA9M            9.8785          28.9406               0.0   

   resp_pmcough_yn_y  resp_diagnosis_yn_y  resp_bronch_yn_y  \
0                0.0                  0.0               0.0   
1                0.0                  0.0               0.0   
2                0.0                  0.0               0.0   
3                0.0                  0.0               0.0   
4                0.0                  0.0               0.0   

   physical_activity1_y  cbcl_scr_syn_internal_t  cbcl_scr_syn_external_t  

In [137]:
# Create respiratory composite (sum of 0/1 indicators across items) and replace the 4 columns with 1 new column
resp_cols = [
    "resp_wheeze_yn_y",
    "resp_pmcough_yn_y",
    "resp_diagnosis_yn_y",
    "resp_bronch_yn_y",
]

existing_resp_cols = [c for c in resp_cols if c in phyhealth_df.columns]
missing_resp_cols = [c for c in resp_cols if c not in phyhealth_df.columns]

if len(existing_resp_cols) == 0:
    if "resp_composite" in phyhealth_df.columns:
        print("Resp columns not present; leaving existing resp_composite as-is.")
    else:
        print("Warning: missing respiratory columns:", missing_resp_cols)
else:
    if missing_resp_cols:
        print("Note: some respiratory columns are missing; composite uses only:", existing_resp_cols)

    resp_numeric = phyhealth_df[existing_resp_cols].apply(pd.to_numeric, errors="coerce")
    phyhealth_df["resp_composite"] = resp_numeric.sum(axis=1, min_count=1)

    # Replace: drop the 4 source columns (where present)
    phyhealth_df = phyhealth_df.drop(columns=[c for c in resp_cols if c in phyhealth_df.columns])

print("Columns now include blood_pressure_mean and resp_composite:")
print(phyhealth_df.columns.tolist())
print("\nresp_composite summary:")
if "resp_composite" in phyhealth_df.columns:
    print(phyhealth_df["resp_composite"].describe(include="all"))

Columns now include blood_pressure_mean and resp_composite:
['src_subject_id', 'mctq_sdweek_calc', 'mctq_msfsc_calc', 'physical_activity1_y', 'cbcl_scr_syn_internal_t', 'cbcl_scr_syn_external_t', 'BMI', 'delta_weight', 'blood_pressure_mean', 'resp_composite']

resp_composite summary:
count    171.000000
mean       0.356725
std        0.665272
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        3.000000
Name: resp_composite, dtype: float64


In [139]:
phyhealth_df_save_path = os.path.join(deriv_dir, "phyhealth-reg-mod.csv")
phyhealth_df.to_csv(phyhealth_df_save_path, index=False)